# Matplotlib vs. Seaborn vs. Plotly

Este notebook grafica **el mismo dataset** (`tips`, propinas en un restaurante) con las tres librerías, para comparar:

- cuánto código hace falta para cada gráfico,
- qué hace cada librería "sola" y qué hay que hacer a mano,
- si el gráfico es estático (imagen) o interactivo.

Para que la comparación sea justa, los tres usan **los mismos colores, títulos y etiquetas**.

**Cómo usarlo:** ejecutá las celdas en orden, de arriba hacia abajo (`Shift + Enter` ejecuta la celda actual y pasa a la siguiente).

## 1. Preparación

Importamos las librerías y cargamos el dataset. `sns.load_dataset` lo descarga de internet la primera vez, así que hace falta conexión.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

tips = sns.load_dataset("tips")

# Traducimos los valores al español (los nombres de columna quedan en inglés)
tips["day"] = tips["day"].cat.rename_categories(
    {"Thur": "Jue", "Fri": "Vie", "Sat": "Sáb", "Sun": "Dom"}
)
tips["time"] = tips["time"].cat.rename_categories({"Lunch": "Almuerzo", "Dinner": "Cena"})

print(f"{len(tips)} filas, {tips.shape[1]} columnas")
tips.head()

Cada fila es una mesa atendida. Las columnas que vamos a usar:

| Columna | Qué es |
|---|---|
| `total_bill` | Total de la cuenta (USD) |
| `tip` | Propina dejada (USD) |
| `day` | Día de la semana (Jue, Vie, Sáb, Dom) |
| `time` | Almuerzo o Cena |
| `size` | Cantidad de personas en la mesa |

### Estilo compartido

Definimos colores y textos **una sola vez** y los reutilizamos en todos los gráficos. Así, si los gráficos se ven distintos, es por la librería y no porque elegimos cosas distintas.

Los colores (azul y naranja) se distinguen bien también para personas con daltonismo.

In [ ]:
AZUL = "#2a78d6"
COLORES_MOMENTO = {"Almuerzo": "#2a78d6", "Cena": "#eb6834"}
ORDEN_DIAS = ["Jue", "Vie", "Sáb", "Dom"]

ETIQUETAS = {
    "total_bill": "Cuenta total (USD)",
    "tip": "Propina (USD)",
    "day": "Día",
    "time": "Momento",
    "size": "Personas",
}

# Matplotlib y Seaborn comparten esta configuración (Seaborn dibuja usando Matplotlib por debajo)
plt.rcParams.update({
    "figure.figsize": (7, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})

---
## 2. Gráfico de dispersión: ¿más cuenta, más propina?

Cada punto es una mesa. El color indica si fue almuerzo o cena.

### Matplotlib
Matplotlib no sabe qué es una "columna de categorías": tenemos que **filtrar los datos a mano** y dibujar un grupo por vez con un `for`.

In [ ]:
fig, ax = plt.subplots()

for momento, color in COLORES_MOMENTO.items():
    datos = tips[tips["time"] == momento]
    ax.scatter(datos["total_bill"], datos["tip"], s=40, color=color, alpha=0.8,
               edgecolor="white", linewidth=0.8, label=momento)

ax.set_title("Propina vs. cuenta total")
ax.set_xlabel(ETIQUETAS["total_bill"])
ax.set_ylabel(ETIQUETAS["tip"])
ax.legend(title=ETIQUETAS["time"])
plt.show()

### Seaborn
Le pasamos la tabla entera y le decimos qué columna va en cada lugar. Con `hue="time"` separa los grupos y arma la leyenda solo.

In [ ]:
fig, ax = plt.subplots()

sns.scatterplot(data=tips, x="total_bill", y="tip", hue="time", palette=COLORES_MOMENTO,
                s=40, alpha=0.8, edgecolor="white", linewidth=0.8, ax=ax)

ax.set(title="Propina vs. cuenta total", xlabel=ETIQUETAS["total_bill"], ylabel=ETIQUETAS["tip"])
ax.legend(title=ETIQUETAS["time"])
plt.show()

### Plotly
Misma idea que Seaborn (tabla + columnas), pero el resultado es **interactivo**: pasá el mouse sobre un punto para ver sus datos, hacé zoom arrastrando, o hacé clic en la leyenda para ocultar un grupo. Con `hover_data` agregamos más columnas al cartelito.

In [ ]:
fig = px.scatter(tips, x="total_bill", y="tip", color="time",
                 color_discrete_map=COLORES_MOMENTO, hover_data=["day", "size"],
                 labels=ETIQUETAS, title="Propina vs. cuenta total",
                 template="simple_white", width=700, height=450)

fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=1, color="white")))
fig.show()

---
## 3. Gráfico de barras: propina promedio por día

### Matplotlib
Matplotlib solo dibuja: **el promedio lo calculamos antes** con pandas (`groupby` + `mean`).

In [ ]:
promedio = tips.groupby("day", observed=True)["tip"].mean()

fig, ax = plt.subplots()
barras = ax.bar(promedio.index.astype(str), promedio.values, color=AZUL, width=0.6)
ax.bar_label(barras, fmt="%.2f", padding=3)  # escribe el valor arriba de cada barra

ax.set(title="Propina promedio por día", xlabel=ETIQUETAS["day"], ylabel="Propina promedio (USD)")
ax.grid(axis="x", visible=False)
plt.show()

### Seaborn
Seaborn **calcula el promedio solo** (`estimator="mean"`). Por defecto también dibuja una barrita de error; la apagamos con `errorbar=None` para que quede igual a los otros.

In [ ]:
fig, ax = plt.subplots()

sns.barplot(data=tips, x="day", y="tip", estimator="mean", errorbar=None,
            color=AZUL, saturation=1, width=0.6, ax=ax)
ax.bar_label(ax.containers[0], fmt="%.2f", padding=3)

ax.set(title="Propina promedio por día", xlabel=ETIQUETAS["day"], ylabel="Propina promedio (USD)")
ax.grid(axis="x", visible=False)
plt.show()

### Plotly
`px.bar` no calcula promedios, así que reutilizamos la tabla `promedio` de la celda de Matplotlib. `text_auto` escribe los valores sobre las barras.

In [ ]:
fig = px.bar(promedio.reset_index(), x="day", y="tip", text_auto=".2f",
             category_orders={"day": ORDEN_DIAS}, color_discrete_sequence=[AZUL],
             labels={**ETIQUETAS, "tip": "Propina promedio (USD)"},
             title="Propina promedio por día", template="simple_white", width=700, height=450)

fig.update_traces(width=0.6, textposition="outside")
fig.show()

---
## 4. Histograma: ¿cómo se reparten los montos de las cuentas?

Un histograma divide los montos en intervalos ("bins") y cuenta cuántas cuentas caen en cada uno.

### Matplotlib

In [ ]:
fig, ax = plt.subplots()

ax.hist(tips["total_bill"], bins=20, color=AZUL, edgecolor="white", linewidth=1)

ax.set(title="Distribución de la cuenta total", xlabel=ETIQUETAS["total_bill"], ylabel="Cantidad de cuentas")
ax.grid(axis="x", visible=False)
plt.show()

### Seaborn
Casi igual a Matplotlib. Un extra: con `kde=True` agrega una curva suave que resume la forma de la distribución (probalo cambiando `False` por `True`).

In [ ]:
fig, ax = plt.subplots()

sns.histplot(data=tips, x="total_bill", bins=20, kde=False,
             color=AZUL, alpha=1, edgecolor="white", linewidth=1, ax=ax)

ax.set(title="Distribución de la cuenta total", xlabel=ETIQUETAS["total_bill"], ylabel="Cantidad de cuentas")
ax.grid(axis="x", visible=False)
plt.show()

### Plotly
Ojo: en Plotly `nbins` es un número **máximo aproximado** de intervalos, así que los bins pueden no coincidir exactamente con los de arriba.

In [ ]:
fig = px.histogram(tips, x="total_bill", nbins=20, color_discrete_sequence=[AZUL],
                   labels=ETIQUETAS, title="Distribución de la cuenta total",
                   template="simple_white", width=700, height=450)

fig.update_traces(marker_line_color="white", marker_line_width=1)
fig.update_layout(yaxis_title="Cantidad de cuentas")
fig.show()

---
## 5. Diagrama de caja (boxplot): cuenta total por día

La caja va del 25% al 75% de los datos, la línea blanca es la mediana, y los puntos sueltos son valores atípicos.

### Matplotlib
Hay que armar **una lista con los datos de cada día** y después pintar cada caja con un `for`.

In [ ]:
datos_por_dia = [tips.loc[tips["day"] == dia, "total_bill"] for dia in ORDEN_DIAS]

fig, ax = plt.subplots()
cajas = ax.boxplot(datos_por_dia, tick_labels=ORDEN_DIAS, patch_artist=True, widths=0.5,
                   medianprops=dict(color="white", linewidth=2))
for caja in cajas["boxes"]:
    caja.set(facecolor=AZUL, edgecolor=AZUL)

ax.set(title="Cuenta total por día", xlabel=ETIQUETAS["day"], ylabel=ETIQUETAS["total_bill"])
ax.grid(axis="x", visible=False)
plt.show()

### Seaborn
Una sola llamada: agrupa por día y respeta el orden de los días automáticamente.

In [ ]:
fig, ax = plt.subplots()

sns.boxplot(data=tips, x="day", y="total_bill", color=AZUL, saturation=1, width=0.5,
            medianprops=dict(color="white", linewidth=2), ax=ax)

ax.set(title="Cuenta total por día", xlabel=ETIQUETAS["day"], ylabel=ETIQUETAS["total_bill"])
ax.grid(axis="x", visible=False)
plt.show()

### Plotly
Pasá el mouse sobre una caja: Plotly muestra mínimo, cuartiles, mediana y máximo sin que tengamos que calcular nada.

In [ ]:
fig = px.box(tips, x="day", y="total_bill", category_orders={"day": ORDEN_DIAS},
             color_discrete_sequence=[AZUL], labels=ETIQUETAS,
             title="Cuenta total por día", template="simple_white", width=700, height=450)
fig.show()

---
## 6. Conclusiones

| | Matplotlib | Seaborn | Plotly |
|---|---|---|---|
| **Qué es** | La base: dibuja lo que le indiques, elemento por elemento | Una capa sobre Matplotlib pensada para estadística | Librería independiente de gráficos web interactivos |
| **Datos de entrada** | Listas / columnas sueltas | La tabla entera + nombres de columnas | La tabla entera + nombres de columnas |
| **Agrupar por categoría** | A mano (filtros y `for`) | Automático (`hue`, `x="day"`) | Automático (`color`, `x="day"`) |
| **Cálculos (promedios, etc.)** | A mano, con pandas | Los hace solo | Depende del gráfico (`px.bar` no, `px.box` sí) |
| **Resultado** | Imagen estática | Imagen estática | Interactivo (zoom, hover, filtros) |
| **Control fino** | Total | Alto (podés usar Matplotlib encima) | Alto, con otra sintaxis |
| **Ideal para** | Gráficos a medida, publicaciones | Explorar datos rápido | Dashboards y presentaciones web |

**En resumen:** Seaborn y Plotly piden menos código para gráficos comunes porque entienden tablas. Matplotlib pide más trabajo pero permite controlar todo, y además es la base sobre la que funciona Seaborn. Plotly es la opción cuando el gráfico se va a ver en una pantalla y conviene poder explorarlo.